## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-07-18-53-43 +0000,nypost,"Russia, Ukraine unleash missile and drone barr...",https://nypost.com/2026/11/07/world-news/russi...
1,2026-11-07-18-52-27 +0000,nyt,Inside the Nightclub Sheltering Haitians From ...,https://www.nytimes.com/video/world/americas/1...
2,2026-11-07-18-52-25 +0000,nyt,A ‘Legend of Zelda’ Game Is Reborn and ’90s No...,https://www.nytimes.com/2026/11/07/business/me...
3,2026-11-07-18-52-08 +0000,startribune,Live: Gophers rally to tie unbeaten UCLA at ha...,https://www.startribune.com/minnesota-gophers-...
4,2026-11-07-18-49-32 +0000,nyt,Islamist Militant Attack Forces Patients to Fl...,https://www.nytimes.com/2026/11/07/world/afric...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2471/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
8,trump,43
288,new,25
96,miller,21
95,stephen,19
155,illinois,18


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
177,2026-11-07-02-00-00 +0000,wsj,President Trump has chosen Stephen Miller—an a...,https://www.wsj.com/politics/elections/stephen...,222
238,2026-11-06-22-18-00 +0000,wsj,Trump said that longtime adviser Stephen Mille...,https://www.wsj.com/politics/elections/trump-n...,181
315,2026-11-06-19-51-00 +0000,wsj,"Stephen Miller, who has been one of Trump’s mo...",https://www.wsj.com/politics/elections/in-tapp...,179
326,2026-11-06-19-07-29 +0000,cbc,Trump names Stephen Miller as new chief of sta...,https://www.cbc.ca/news/world/trump-susie-wile...,173
301,2026-11-06-20-11-45 +0000,bbc,Trump names Stephen Miller as new White House ...,https://www.bbc.co.uk/news/articles/cm9qzz5x9g...,167


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
177,222,2026-11-07-02-00-00 +0000,wsj,President Trump has chosen Stephen Miller—an a...,https://www.wsj.com/politics/elections/stephen...
85,106,2026-11-07-13-00-05 +0000,chitri,Bill Foster vs Jeff Walter: How Illinois’ 12th...,https://www.chicagotribune.com/2026/11/07/bill...
315,66,2026-11-06-19-51-00 +0000,wsj,"Stephen Miller, who has been one of Trump’s mo...",https://www.wsj.com/politics/elections/in-tapp...
260,58,2026-11-06-21-48-51 +0000,nypost,Dossier urging Democrats to go to war with cor...,https://nypost.com/2026/11/06/business/dossier...
56,45,2026-11-07-16-09-00 +0000,wsj,Is AI a white-collar job killer? Strong hiring...,https://www.wsj.com/economy/jobs/is-ai-taking-...
102,42,2026-11-07-11-40-00 +0000,wsj,New Zealand’s ruling conservatives led by a sl...,https://www.wsj.com/world/oceania/new-zealands...
7,42,2026-11-07-18-46-13 +0000,chitri,"Gunman opens fire in Polish city of Krakow, ki...",https://www.chicagotribune.com/2026/11/07/gunm...
314,42,2026-11-06-19-52-37 +0000,nypost,Pete Hegseth’s future in doubt as Republicans ...,https://nypost.com/2026/11/06/us-news/pete-heg...
151,42,2026-11-07-07-27-44 +0000,bbc,Meghan's father Thomas Markle under house arre...,https://www.bbc.co.uk/news/articles/cqgkvv8m8p...
169,41,2026-11-07-03-00-00 +0000,wsj,Elon Musk-run companies like SpaceX and Tesla ...,https://www.wsj.com/politics/policy/elon-musk-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
